<img src="../Images/DSC_Logo.png" style="width: 400px;">

# Basics of Quantitative Data Analysis in Python
# - First Analytical Steps

In this notebook, we move from cleaned data to first analytical steps. These often include:

- **summarizing** variables,
- **comparing** groups,
- **visualizing** patterns,
- **testing** simple hypotheses,
- **exploring relationships** between variables,
- and **fitting models**.

For each of these topics, the notebook provides basic Python code examples and shows how they can support quantitative data analysis.

In [ ]:
# Install all required libraries for this notebook:

!pip install -q -r ../requirements_B.txt   

print("The libraries have been installed.")

--

We start by importing libraries needed for this notebook and loading the datasets. 

Besides `pandas`, `numpy`, and `matplotlib`, we also use three widely used libraries for statistics and modeling: `statsmodels`, `scikit-learn`, and `scipy`:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Tools for statistics and modeling:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm

from scipy import stats

The examples in this notebook are intentionally small and use three datasets that were already introduced in B01 and B02, which we again load as `pandas` dataframes:
1. the famous Iris dataset
2. the World Happiness Report dataset(https://www.worldhappiness.report/ed/2024/)
3. the [NOAA temperature anomaly time series dataset](https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/global/time-series)

In [ ]:
Iris = pd.read_csv("../Data/DataB/Iris.csv")
report = pd.read_excel("../Data/DataB/World-happiness-report-2024.xlsx")
noaa = pd.read_csv("../Data/DataB/NOAA_time_series.csv", skiprows=4)

## 1. Exploring a Single Variable

A first step in data analysis is often to **summarize variables with descriptive statistics**. 

- `.describe()` was already introduced in previous notebooks. Here we use it again in an analytical context, for example, to get a quick overview of the numeric columns in the Iris dataset, including count, mean, standard deviation, minimum, maximum, and quartiles:

In [ ]:
Iris.describe()

- We can also calculate specific statistics directly for one variable:

In [ ]:
print(Iris["SepalLengthCm"].mean())     # average (mean) value
print(Iris["SepalLengthCm"].median())   # middle (median) value
print(Iris["SepalLengthCm"].min())      # smallest (minimum) value
print(Iris["SepalLengthCm"].max())      # largest (maximum) value
print(Iris["SepalLengthCm"].std())      # spread around the mean (standard deviation)

Summary statistics provide a first overview, but **plots** are often needed to understand **patterns in the data** more fully, such as value distributions, trends, and relationships.

- **Line plots** are especially useful for showing how values change over time. In the temperature anomaly time series, a line plot gives a first visual impression of how anomalies develop across years:

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(noaa["Year"], noaa["Anomaly"], color="blue", linestyle="-")
plt.title("Temperature Anomaly Over Time")
plt.xlabel("Year")
plt.ylabel("Anomaly")
plt.show()

- A prominent example of showing values across time with bars is a warming-stripes-style **bar plot**. Here, each bar represents one time step and is colored according to the temperature anomaly value in that time step. This makes long-term patterns in the data visible very quickly. Below, we use annual mean temperature anomalies from the NOAA dataset and color the bars, where blue tones indicate lower anomalies and red tones indicate higher anomalies:

In [ ]:
# Normalize anomaly values for coloring
norm = mcolors.Normalize(
    vmin=noaa["Anomaly"].min(),
    vmax=noaa["Anomaly"].max()
)
cmap = plt.get_cmap("RdBu_r")
colors = [cmap(norm(value)) for value in noaa["Anomaly"]]

# Plot
plt.figure(figsize=(10, 4))
plt.bar(noaa["Year"], noaa["Anomaly"], color=colors, width=1.0)
plt.title("Annual Mean Temperature Anomalies")
plt.xlabel("Year")
plt.ylabel("Anomaly (°C)")
plt.tight_layout()
plt.show()

- See whether the values in the Iris dataset are concentrated in one range or spread across several ranges in a **histogram**:

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(Iris["PetalLengthCm"], edgecolor="black")
plt.title("Distribution of Petal Length")
plt.xlabel("Petal Length (cm)")
plt.ylabel("Count")
plt.show()

---

### **Exercise 1:**

Create a new dataframe containing the 20 countries with the highest `Ladder score` in report, and visualize their scores in a bar plot with matplotlib.

Hints:
- First sort the dataframe by `Ladder score` in descending order (see B02).
- Then select the first 20 rows.
- Use `plt.bar(...)` in a similar way to the earlier bar plot example.

---

## 2. Comparing Data Across Groups

Many research questions are not only about single variables, but about comparisons between groups.

It is often useful to investigate **how many observations** fall into each group/category.

- Count how many samples belong to each species:

In [ ]:
Iris["Species"].value_counts()

- Visualize the number of observations per species using a **bar plot**:

In [ ]:
Iris["Species"].value_counts().plot(kind="bar")
plt.title("Number of Observations per Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.show()

It is often useful to derive **aggregated information** for each group.

- Here, `groupby()` follows a simple logic: 1. split the data into groups, 2. apply a calculation within each group, and 3. compare the results. For example, we can compare average measurements of different species in the Iris dataset:

In [ ]:
Iris.groupby("Species").mean(numeric_only=True)

- We can also focus on one variable:

In [ ]:
Iris.groupby("Species")["SepalLengthCm"].mean()

- A **boxplot** is useful for visually comparing data distributions (including statistics such as median values) across groups:


In [ ]:
Iris.boxplot(column="SepalLengthCm", by="Species")
plt.title("Sepal Length by Species")
plt.suptitle("")
plt.xlabel("Species")
plt.ylabel("Sepal Length (cm)")
plt.show()

---

### **Exercise 2:**

Compare scores in the World Happiness Report across regions.

Write code that groups the data by region (column: `Regional indicator`) and calculates the mean `Ladder score` for each group.

---

**Statistical tests** can also be useful for comparing groups. They help assess whether an observed difference is likely to be due to random variation or reflects a more systematic difference in the data.

As an example, we compare the mean `SepalLengthCm` values of two Iris species using an independent **two-sample t-test**.

- First, select the two groups:

In [ ]:
setosa = Iris.loc[Iris["Species"] == "Iris-setosa", "SepalLengthCm"]
versicolor = Iris.loc[Iris["Species"] == "Iris-versicolor", "SepalLengthCm"]

print(setosa.head())
print(versicolor.head())

- Next, we run the t-test:

In [ ]:
t_stat, p_value = stats.ttest_ind(setosa, versicolor)

print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

- The t-statistic measures the size of the difference relative to variation in the data.

- The p-value indicates how compatible the observed difference is with the assumption that the two groups have the same mean. A simple rule of thumb is:
    - if the p-value is small (commonly below 0.05), the observed difference is often interpreted as statistically significant
    - if the p-value is larger, the observed difference may be due to random variation

In [ ]:
if p_value < 0.05:
    print("The difference in mean sepal length between the two species is statistically significant.")
else:
    print("The difference in mean sepal length between the two species is not statistically significant.")

## 3. Relationships Between Continuous Variables

In many analyses, we want to understand whether two variables seem to vary together.

A **scatter plot** is a simple way to visualize the relationship between two continuous variables.

- For example, sepal length and sepal width in the Iris dataset:

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(Iris["SepalLengthCm"], Iris["SepalWidthCm"])
plt.title("Sepal Length and Sepal Width")
plt.xlabel("Sepal Length (cm)")
plt.ylabel("Sepal Width (cm)")
plt.show()

- To add one grouping variable, we can plot each species separately using a for-loop inside the plotting code:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for species in Iris["Species"].unique():
    subset = Iris[Iris["Species"] == species]
    ax.scatter(subset["SepalLengthCm"], subset["SepalWidthCm"], label=species)

ax.set_title("Sepal Length and Sepal Width by Species")
ax.set_xlabel("Sepal Length (cm)")
ax.set_ylabel("Sepal Width (cm)")
ax.legend()
plt.show()

**Correlation** provides a simple numerical summary of the relationship between continuous variables. 

- For example, we can calculate the correlation coefficient between petal length and petal width. By default, `.corr()` in `pandas` computes the Pearson correlation. That means it measures the strength of a linear relationship between two variables.

In [ ]:
Iris["PetalLengthCm"].corr(Iris["PetalWidthCm"])

- We can also calculate correlations for several variables at once (correlation matrix):

In [ ]:
corr_matrix = Iris[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]].corr()

print(corr_matrix)

- A simple way is to show the correlation matrix with `imshow()`:

In [ ]:
# Plot the correlation matrix
plt.figure(figsize=(6, 4))
plt.imshow(corr_matrix, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")

# Add axis labels
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 4. Mathematical Operations and Modeling

Let’s see how to run basic **calculations** with existing data in columns. 

- Often, we want to derive a new variable from existing measurements. We can, for example, convert temperature values from degrees Celsius to Kelvin by adding 273.15:

In [ ]:
noaa["Anomaly_K"] = noaa["Anomaly"] + 273.15
print(noaa["Anomaly_K"])

- If the conversion factor were stored in another column instead of being added as a constant value, the calculation would work in the same way, but row by row using the values from that column:

In [ ]:
noaa["ConversionFactor"] = 273.15
noaa["Anomaly_K"] = noaa["Anomaly"] + noaa["ConversionFactor"]
print(noaa["Anomaly_K"])

---

### **Exercise 3:**

Calculate the area of the petals of the Iris flower in a new column (a simple approximation). Then select only the species Iris-versicolor and calculate the mean petal area for this species.

---

A simple **modeling** approach to analyzing relationships in data is to fit a **linear regression model**. Linear regression describes how one variable changes in relation to another.

Below, we use linear regression in two closely related ways:

- first, as **trend analysis**, where time (`Year`) is the predictor variable,
- second, as a more general **regression model** between two measured variables.

Both `statsmodels` and `scikit-learn` can be used for linear regression in Python. Below, `statsmodels` is used first because it provides a detailed statistical summary. Afterwards, `scikit-learn` is used to show a more prediction-oriented workflow with training data, test data, and performance measures.

--

- First, we **fit a trend line** that represents the overall linear trend in the Anomaly time series.

1. Define predictor and target data:

In [ ]:
# Predictor variable: year
X = noaa["Year"]  

# Add an intercept column ("const") to the model: This turns X into a dataframe with two columns: const and Year
X = sm.add_constant(X)

# Response variable: temperature anomaly
y = noaa["Anomaly"]

2. Fit a simple linear regression model:

In [ ]:
trend_model = sm.OLS(y, X).fit()

3. Show a detailed summary of the regression results:

In [ ]:
print(trend_model.summary())

4. Save the model's predicted values as a new column in the dataframe:

In [ ]:
noaa["Trend"] = trend_model.fittedvalues  

5. Plot the observed data together with the fitted linear trend:

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(noaa["Year"], noaa["Anomaly"], color="blue", linestyle="-", label="Observed data")
plt.plot(noaa["Year"], noaa["Trend"], color="red", linewidth=2, label="Linear trend")
plt.title("Temperature Anomaly Over Time")
plt.xlabel("Year")
plt.ylabel("Anomaly")
plt.legend()
plt.show()

- Next, we use the same basic method again, but now not for a time trend. Instead, we **model the relationship** between two measured variables in the Iris dataset. In this example, `PetalLengthCm` is used to predict `PetalWidthCm`. This is again a linear regression model, but now the goal is not to describe change over time, but rather the relationship between two variables.


1. Define predictor and target data:

In [ ]:
# Predictor variable
X = Iris[["PetalLengthCm"]]     # Double brackets keep X as a DataFrame with one column

# Response variable
y = Iris["PetalWidthCm"]       

2. Separate the dataset into one part for training the model and one part for testing it. The model learns from the training data and is evaluated on the test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42                # 30% of the data are kept for testing; random_state=42 makes the split reproducible
)

3. Fit the model, i.e., estimate a linear relationship between the predictor variable and the target variable based on the training data:

In [ ]:
# Create a linear regression model
regressor = LinearRegression()

# Fit the model to the training data (the model learns the relationship between petal length and petal width)
regressor.fit(X_train, y_train)

4. Make predictions, i.e., calculate the target variable by using the fitted model to predict petal width for the test data:

In [ ]:
y_pred = regressor.predict(X_test)

5. Evaluate the model, i.e., calculate model performance measures:

In [ ]:
# Calculate measures
mse = mean_squared_error(y_test, y_pred)    # average squared prediction error
r2 = r2_score(y_test, y_pred)               # proportion of explained variation

# Print the evaluation results
print(f"Mean squared error: {mse}")      
print(f"R²: {r2}")

6. Print the parameters of the fitted regression line:

In [ ]:
print(f"Intercept: {regressor.intercept_}")
print(f"Coefficient: {regressor.coef_[0]}")

7. Plot the observed data together with the fitted regression line:

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(Iris["PetalLengthCm"], Iris["PetalWidthCm"], alpha=0.7, label="Observed data")
plt.plot(Iris["PetalLengthCm"], regressor.predict(Iris[["PetalLengthCm"]]), color="red", label="Fitted line")
plt.title("Petal Length and Petal Width")
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.legend()
plt.show()

---

### **Exercise 4: Optional Challenge**

**Option 1: Extend the Iris analysis**  
Carry out one additional analysis, visualization, or simple model with the Iris dataset that was not shown in the notebook. Add short comments to your code and briefly describe what you found. You can find many tutorials with Iris dataset online.

**Option 2: Try your own dataset**  
Apply parts of B01–B03 to one of your own datasets: load it, inspect it, preprocess it if needed, and create at least one summary, visualization, or simple relationship analysis. Add short comments to explain your steps.

---

## Key Takeaways

In this notebook, you learned that:

- basic quantitative analysis includes summarizing, comparing, visualizing, and relating variables,
- common tools include descriptive statistics, grouped summaries, plots, correlation, statistical tests, and model fitting,
- analyses become easier to understand when each step is linked to a clear question and the output is interpreted in context.